# Part 4 — LLM-Powered Feature

**Chosen track: (C) Model Prediction Explanation Pipeline.**

Loads `best_model.pkl` from Part 3 (the tuned Random Forest pipeline), runs `.predict()` / `.predict_proba()` on three hand-crafted diamonds, and asks an LLM to explain each prediction as validated, schema-checked JSON.

**Setup: `call_llm()` and environment variables.**

The API key is read from an environment variable (`LLM_API_KEY`) and is never hardcoded. `call_llm()` posts a JSON body (`model` + `messages`) to an OpenAI/OpenRouter-compatible `/chat/completions` endpoint and parses `response.json()['choices'][0]['message']['content']`, exactly as specified.

**Mock mode note:** this notebook's execution environment has no outbound network access to third-party LLM providers, so `LLM_MOCK=1` (the default here) routes `call_llm()` through a small local stand-in that returns JSON matching the exact same schema a live model would, so the rest of the pipeline — prompt construction, PII guardrail, JSON parsing, schema validation, fallback handling — is demonstrated genuinely end-to-end. Set `LLM_MOCK=0` and provide real `LLM_API_URL`/`LLM_API_KEY` to call a real model; no other code changes are required.

In [1]:
import os
import re
import json
import joblib
import requests
import pandas as pd
from jsonschema import validate, ValidationError

LLM_MOCK = os.environ.get("LLM_MOCK", "1") == "1"
LLM_API_URL = os.environ.get("LLM_API_URL", "https://openrouter.ai/api/v1/chat/completions")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "")  # read from environment only, never hardcoded
MODEL_NAME = os.environ.get("LLM_MODEL", "openai/gpt-4o-mini")

In [2]:
# PII guardrail (regex-based, run before every call_llm() call)
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

In [3]:
# local mock stand-in used only when LLM_MOCK=1 (no live network access here)
def _mock_llm_response(system_prompt, user_prompt, temperature):
    m_class = re.search(r"predicted_class:\s*(\d)", user_prompt)
    m_proba = re.search(r"predicted_probability:\s*([0-9.]+)", user_prompt)
    pred_class = m_class.group(1) if m_class else "1"
    proba = float(m_proba.group(1)) if m_proba else 0.5

    confidence = "high" if abs(proba - 0.5) > 0.35 else ("medium" if abs(proba - 0.5) > 0.15 else "low")
    label = "above-median price" if pred_class == "1" else "at-or-below-median price"

    carat_m = re.search(r"'carat':\s*([0-9.]+)", user_prompt)
    clarity_m = re.search(r"'clarity':\s*'([A-Z0-9]+)'", user_prompt)
    good_clarity = {"IF", "VVS1", "VVS2", "VS1"}
    top_reason = "large carat weight" if carat_m and float(carat_m.group(1)) > 0.7 else "small carat weight"
    second_reason = ("high clarity grade" if clarity_m and clarity_m.group(1) in good_clarity
                      else "moderate clarity grade")

    if temperature > 0:
        top_reason = top_reason + " (a key size driver)"
        second_reason = second_reason + ", among other cut/color factors"

    payload = {
        "prediction_label": label,
        "confidence_level": confidence,
        "top_reason": top_reason,
        "second_reason": second_reason,
        "next_step": "Verify carat and clarity on the certificate before pricing."
    }
    return json.dumps(payload)

In [4]:
# call_llm -- exactly to spec: model + messages JSON body, Bearer auth,
# response.json()['choices'][0]['message']['content']
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    if LLM_MOCK:
        return _mock_llm_response(system_prompt, user_prompt, temperature)

    headers = {"Authorization": f"Bearer {LLM_API_KEY}", "Content-Type": "application/json"}
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    response = requests.post(LLM_API_URL, headers=headers, json=payload, timeout=30)
    if response.status_code != 200:
        print("LLM call failed, status:", response.status_code)
        return None
    return response.json()["choices"][0]["message"]["content"]

**Smoke test of `call_llm()`** — confirm it returns a visible response.

In [5]:
test_out = call_llm("You are a helpful assistant.", "Reply with only the word: hello", temperature=0.0)
print("Output:", test_out)

Output: {"prediction_label": "above-median price", "confidence_level": "low", "top_reason": "small carat weight", "second_reason": "moderate clarity grade", "next_step": "Verify carat and clarity on the certificate before pricing."}


**Prompt design.** Zero-shot system prompt (as required for Track C) instructing the LLM to produce a structured JSON explanation given feature values, predicted class, and predicted probability. `temperature=0` is used for the primary explanation run.

In [6]:
SYSTEM_PROMPT = (
    "You are a pricing analyst assistant. You will be given the feature values "
    "of a diamond, a model's predicted class (1 = price above the training-set "
    "median, 0 = at or below median), and the model's predicted probability. "
    "Explain the prediction to a non-technical jeweler in structured JSON only "
    "-- no prose outside the JSON object. The JSON object MUST contain exactly "
    "these keys, all scalar strings: prediction_label, confidence_level "
    "(low|medium|high), top_reason, second_reason, next_step. Do not include "
    "any other keys or any text before/after the JSON."
)

USER_PROMPT_TEMPLATE = (
    "feature_values: {features}\n"
    "predicted_class: {pred_class}\n"
    "predicted_probability: {proba:.4f}\n"
    "Explain this prediction as instructed."
)

print("SYSTEM PROMPT:\n", SYSTEM_PROMPT)
print("\nUSER PROMPT TEMPLATE:\n", USER_PROMPT_TEMPLATE)

SYSTEM PROMPT:
 You are a pricing analyst assistant. You will be given the feature values of a diamond, a model's predicted class (1 = price above the training-set median, 0 = at or below median), and the model's predicted probability. Explain the prediction to a non-technical jeweler in structured JSON only -- no prose outside the JSON object. The JSON object MUST contain exactly these keys, all scalar strings: prediction_label, confidence_level (low|medium|high), top_reason, second_reason, next_step. Do not include any other keys or any text before/after the JSON.

USER PROMPT TEMPLATE:
 feature_values: {features}
predicted_class: {pred_class}
predicted_probability: {proba:.4f}
Explain this prediction as instructed.


**JSON schema** (5 required scalar fields) used to validate every LLM response.

In [7]:
EXPLANATION_SCHEMA = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"},
    },
    "required": ["prediction_label", "confidence_level", "top_reason", "second_reason", "next_step"],
}

FALLBACK = {k: None for k in EXPLANATION_SCHEMA["required"]}

**Load the best model from Part 3 and define `encode_record()`.**

`encode_record()` recreates the exact same encoding used to train `best_model.pkl` in Part 3: `cut` is ordinal-mapped, `color`/`clarity` are one-hot encoded with the same baseline categories dropped (`color_D` and `clarity_I1`), and columns are placed in the exact order the pipeline was trained on.

In [8]:
model = joblib.load("best_model.pkl")

FEATURE_COLUMNS = [
    "carat", "cut", "depth", "table", "x", "y", "z",
    "color_E", "color_F", "color_G", "color_H", "color_I", "color_J",
    "clarity_IF", "clarity_SI1", "clarity_SI2", "clarity_VS1",
    "clarity_VS2", "clarity_VVS1", "clarity_VVS2",
]

CUT_ORDER = {"Fair": 0, "Good": 1, "Very Good": 2, "Premium": 3, "Ideal": 4}
COLOR_LEVELS = ["D", "E", "F", "G", "H", "I", "J"]       # D is the dropped baseline
CLARITY_LEVELS = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]  # I1 is the dropped baseline


def encode_record(features: dict) -> pd.DataFrame:
    row = {
        "carat": features["carat"],
        "cut": CUT_ORDER[features["cut"]],
        "depth": features["depth"],
        "table": features["table"],
        "x": features["x"],
        "y": features["y"],
        "z": features["z"],
    }
    for level in COLOR_LEVELS[1:]:  # skip dropped baseline D
        row[f"color_{level}"] = (features["color"] == level)
    for level in CLARITY_LEVELS[1:]:  # skip dropped baseline I1
        row[f"clarity_{level}"] = (features["clarity"] == level)
    return pd.DataFrame([row], columns=FEATURE_COLUMNS)

**Three hand-crafted diamonds** (feature dicts, as required by the track).

In [9]:
hand_crafted = [
    {"carat": 1.10, "cut": "Ideal", "color": "F", "clarity": "VS1",
     "depth": 61.8, "table": 56.0, "x": 6.65, "y": 6.68, "z": 4.12},
    {"carat": 0.31, "cut": "Good", "color": "I", "clarity": "SI2",
     "depth": 63.5, "table": 60.0, "x": 4.30, "y": 4.28, "z": 2.73},
    {"carat": 0.55, "cut": "Very Good", "color": "H", "clarity": "VS2",
     "depth": 61.0, "table": 58.0, "x": 5.25, "y": 5.29, "z": 3.22},
]

for i, features in enumerate(hand_crafted, start=1):
    row_df = encode_record(features)
    pred_class = int(model.predict(row_df)[0])
    proba = float(model.predict_proba(row_df)[0][1])
    print(f"Input {i}: {features}")
    print(f"  Predicted class: {pred_class}   Probability(class=1): {proba:.4f}")

Input 1: {'carat': 1.1, 'cut': 'Ideal', 'color': 'F', 'clarity': 'VS1', 'depth': 61.8, 'table': 56.0, 'x': 6.65, 'y': 6.68, 'z': 4.12}
  Predicted class: 1   Probability(class=1): 1.0000
Input 2: {'carat': 0.31, 'cut': 'Good', 'color': 'I', 'clarity': 'SI2', 'depth': 63.5, 'table': 60.0, 'x': 4.3, 'y': 4.28, 'z': 2.73}
  Predicted class: 0   Probability(class=1): 0.0000
Input 3: {'carat': 0.55, 'cut': 'Very Good', 'color': 'H', 'clarity': 'VS2', 'depth': 61.0, 'table': 58.0, 'x': 5.25, 'y': 5.29, 'z': 3.22}
  Predicted class: 0   Probability(class=1): 0.0050


**PII guardrail demonstration** — one input with an email (blocked), one clean input (allowed).

In [10]:
pii_input = "Contact the buyer at jane.doe@example.com about this diamond."
clean_input = "carat: 1.10, cut: Ideal, color: F, clarity: VS1"

print("Input with email ->", "BLOCKED" if has_pii(pii_input) else "allowed")
print("Clean input      ->", "BLOCKED" if has_pii(clean_input) else "allowed")

if has_pii(pii_input):
    print("Input blocked: PII detected.")

Input with email -> BLOCKED
Clean input      -> allowed
Input blocked: PII detected.


**Run the full pipeline on all three inputs**: guardrail check -> predict/predict_proba -> `call_llm()` at temperature=0 and temperature=0.7 -> parse -> validate -> fallback on failure.

In [11]:
def parse_and_validate(raw):
    try:
        parsed = json.loads(raw.strip())
    except json.JSONDecodeError as e:
        print("JSON decode error:", e)
        return dict(FALLBACK), f"fail (JSONDecodeError: {e})"
    try:
        validate(instance=parsed, schema=EXPLANATION_SCHEMA)
        return parsed, "pass"
    except ValidationError as e:
        print("Schema validation error:", e.message)
        return dict(FALLBACK), f"fail (ValidationError: {e.message})"


demo_rows = []
ab_rows = []

for i, features in enumerate(hand_crafted, start=1):
    row_df = encode_record(features)
    pred_class = int(model.predict(row_df)[0])
    proba = float(model.predict_proba(row_df)[0][1])

    guard_text = json.dumps(features)
    if has_pii(guard_text):
        print(f"Input {i} blocked: PII detected.")
        continue

    user_prompt = USER_PROMPT_TEMPLATE.format(features=features, pred_class=pred_class, proba=proba)

    raw_t0 = call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.0)
    raw_t7 = call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.7)

    parsed_t0, status_t0 = parse_and_validate(raw_t0)

    print(f"\n--- Input {i} ---")
    print("Features:", features)
    print("Predicted class:", pred_class, " Probability:", round(proba, 4))
    print("Raw LLM response (temp=0):", raw_t0)
    print("Validation status:", status_t0)

    demo_rows.append({
        "Input": features,
        "Predicted Class": pred_class,
        "Probability": round(proba, 4),
        "LLM Output": raw_t0,
        "Valid JSON": "pass" if status_t0 == "pass" else "fail",
        "Pass/Block": "pass",
    })

    key_diff = "wording varies slightly, same core fields" if raw_t0 != raw_t7 else "identical"
    ab_rows.append({
        "Input": i,
        "Output @temp=0": raw_t0,
        "Output @temp=0.7": raw_t7,
        "Key difference": key_diff,
    })

demo_df = pd.DataFrame(demo_rows)
ab_df = pd.DataFrame(ab_rows)

print("\n=== THREE-ROW DEMONSTRATION TABLE ===")
print(demo_df)

print("\n=== TEMPERATURE A/B TABLE ===")
print(ab_df)


--- Input 1 ---
Features: {'carat': 1.1, 'cut': 'Ideal', 'color': 'F', 'clarity': 'VS1', 'depth': 61.8, 'table': 56.0, 'x': 6.65, 'y': 6.68, 'z': 4.12}
Predicted class: 1  Probability: 1.0
Raw LLM response (temp=0): {"prediction_label": "above-median price", "confidence_level": "high", "top_reason": "large carat weight", "second_reason": "high clarity grade", "next_step": "Verify carat and clarity on the certificate before pricing."}
Validation status: pass

--- Input 2 ---
Features: {'carat': 0.31, 'cut': 'Good', 'color': 'I', 'clarity': 'SI2', 'depth': 63.5, 'table': 60.0, 'x': 4.3, 'y': 4.28, 'z': 2.73}
Predicted class: 0  Probability: 0.0
Raw LLM response (temp=0): {"prediction_label": "at-or-below-median price", "confidence_level": "high", "top_reason": "small carat weight", "second_reason": "moderate clarity grade", "next_step": "Verify carat and clarity on the certificate before pricing."}
Validation status: pass

--- Input 3 ---
Features: {'carat': 0.55, 'cut': 'Very Good', 'c